# Chapter 20 — Numerical Procedures

Every method in the chapter, on one contract: a six-month put, $S_0 = K = 50$,
$r = 5\%$, $\sigma = 30\%$. Keeping the contract fixed is what makes the methods
comparable.

| Section | Method | Example |
|---|---|---|
| 1 | Binomial tree, and its oscillation | Table 20.2 |
| 2 | Control variate on an American put | 20.1 |
| 3 | A cash dividend done three ways | 20.2 |
| 4 | Monte Carlo convergence | 20.3 |
| 5 | The control variate worth 58 million paths | 20.6 |
| 6 | A relative-TSR award | 20.7 |
| 7 | Longstaff–Schwartz, step by step | 20.8, 20.9 |
| 8 | Explicit instability | 20.10 |
| 9 | Implicit against Crank–Nicolson | 20.11 |
| 10 | Greeks from the tree and from the grid | 20.12 |

The deterministic figures (trees, dividends, finite differences) reproduce
exactly. The simulated ones carry their own seed, so those are reported with a
standard error and compared in standard errors rather than digit by digit.


In [ ]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm

N = norm.cdf
S0, K, r, SIG, T = 50.0, 50.0, 0.05, 0.30, 0.5


def show(label, got, printed, tol=6e-5):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-42s %14.6f   book %12.6f   %s" % (label, got, printed, flag))


def bs_put(S, K, T, r, sig):
    d1 = (log(S / K) + (r + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    return K * exp(-r * T) * N(-(d1 - sig * sqrt(T))) - S * N(-d1)


EXACT = bs_put(S0, K, T, r, SIG)
show("European put, Black-Scholes", EXACT, 3.582934)


## 1. The binomial tree, and its oscillation — Table 20.2

The error does not fall smoothly. It oscillates with the parity of $n$, because
whether a node sits on the strike changes with $n$. Everything in the next two
sections follows from that.

In [ ]:
def crr(n, american=False, S=S0, K=K, T=T, r=r, sig=SIG, div=None, t_div=None):
    """CRR tree. With (div, t_div) the tree runs on the displaced spot and the
    exercise test uses the actual stock price, which is the escrowed-dividend
    treatment of Section 20.5."""
    S_tree = S - div * exp(-r * t_div) if div else S
    dt = T / n
    u = exp(sig * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    j = np.arange(n + 1)
    v = np.maximum(K - S_tree * u ** (2 * j - n), 0.0)
    for i in range(n - 1, -1, -1):
        j = np.arange(i + 1)
        S_i = S_tree * u ** (2 * j - i)
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if american:
            t = i * dt
            S_actual = S_i + (div * exp(-r * (t_div - t)) if div and t < t_div else 0.0)
            v = np.maximum(v, K - S_actual)
    return float(v[0])


print("%6s %12s %12s" % ("n", "tree", "error"))
for n, want in ((5, 3.789185), (10, 3.479418), (11, 3.675951), (20, 3.530789),
                (21, 3.631453), (50, 3.561990), (100, 3.572449), (101, 3.592980),
                (200, 3.577688), (201, 3.587979), (500, 3.580835), (1000, 3.581884)):
    v = crr(n)
    print("%6d %12.6f %+12.6f    book %.6f" % (n, v, v - EXACT, want))

## 2. The control variate on an American put — Example 20.1

Price the American and the European on the *same* tree, and correct the American
by the European's known error. What the control variate removes is what the two
problems **share**, which is why it is extraordinary on odd $n$ and does little on
even $n$. The benchmark is the same recipe run far enough: corrected trees at 2,000
and 4,000 steps, extrapolated between them.

In [ ]:
def corrected_tree(n):
    """American tree value plus the tree's error on the European."""
    return crr(n, american=True) + (EXACT - crr(n))


# The benchmark: the correction, then Richardson between two even step counts,
# which share an offset at the money (Section 20.3's production recipe).
AMER = 2 * corrected_tree(4000) - corrected_tree(2000)
show("American put, benchmark", AMER, 3.69702)
show("early exercise premium", AMER - EXACT, 0.114, tol=5e-3)

print()
print("%5s %12s %12s %12s %12s" % ("n", "plain", "error", "corrected", "error"))
for n in (50, 51, 100, 101, 200, 201):
    plain = crr(n, american=True)
    corrected = corrected_tree(n)
    print("%5d %12.5f %+12.5f %12.5f %+12.5f" % (n, plain, plain - AMER, corrected, corrected - AMER))
print()
for n in (200, 201):
    print("n = %d: the correction shrinks the error by a factor of %.1f"
          % (n, abs((crr(n, american=True) - AMER) / (corrected_tree(n) - AMER))))

## 3. A cash dividend done three ways — Example 20.2

Build the tree on $S^{*} = S_0 - D e^{-rt_d}$ and test exercise against the
*actual* stock price. Ignoring the dividend is catastrophic and obvious;
subtracting the undiscounted dividend is a small, one-directional error that
scales with rate and time.

In [ ]:
D, T_DIV = 2.00, 0.25
S_star = S0 - D * exp(-r * T_DIV)
show("displaced spot S*", S_star, 48.0248, tol=1e-5)

for n in (200, 2000, 20001):
    print("  n=%6d   %.4f" % (n, crr(n, american=True, div=D, t_div=T_DIV)))

correct = crr(20001, american=True, div=D, t_div=T_DIV)
euro = bs_put(S_star, K, T, r, SIG)
print()
show("American, displaced tree (correct)", correct, 4.6098, tol=2e-4)
show("European, Black-Scholes on S*", euro, 4.4689, tol=1e-4)
show("dividend ignored", AMER, 3.6970, tol=2e-4)
show("spot reduced by the full 2.00", crr(20001, american=True, S=48.0), 4.6385, tol=2e-4)
print()
print("dividend ignored:      %+.1f%%" % (100 * (AMER / correct - 1)))
print("full 2.00 subtracted:  %+.1f%%" % (100 * (crr(20001, american=True, S=48.0) / correct - 1)))
show("early exercise premium with the dividend", correct - euro, 0.141, tol=2e-2)

### The same dividend paid three ways

A *continuous yield* goes into the drift. A *discrete proportional*
dividend multiplies every node at $t_d$ by the same $1-\delta$, so the
lattice is rescaled rather than distorted and no displacement is needed.
Only a *cash* dividend breaks recombination, because subtracting a
constant moves nodes by different proportions.

Holding the amount fixed at 4% of today's price shows what each choice is
worth: the form of the payment moves the value by 0.3%, its timing by more than 3%.


In [ ]:
# The same 4% dividend paid three ways. A proportional dividend's present
# value is exactly delta * S0, whatever the price does, because
# exp(-r t) S_t is a martingale; 2.00 of cash at t_d is worth less.
DELTA = 0.04
q_equiv = -log(1 - DELTA) / T


def crr_scaled(n, delta=0.0, q=0.0, american=True):
    """Every node from t_div onward carries the same factor 1 - delta, so
    the lattice is rescaled rather than distorted and still recombines.
    With q instead, the dividend is spread smoothly through the drift."""
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp((r - q) * dt) - d) / (u - d)
    disc = exp(-r * dt)

    def spot(i):
        s = S0 * u ** (2 * np.arange(i + 1) - i)
        return s * (1 - delta) if i * dt >= T_DIV else s

    v = np.maximum(K - spot(n), 0.0)
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if american:
            v = np.maximum(v, K - spot(i))
    return v[0]


show("PV of the 2.00 cash dividend", D * exp(-r * T_DIV), 1.9752, tol=1e-4)
show("PV of the 4% proportional dividend", DELTA * S0, 2.0000, tol=1e-9)
show("equivalent continuous yield, %", 100 * q_equiv, 8.16, tol=1e-3)
print()

# Both forms leave the same terminal distribution, 0.96 S0 exp(...), so one
# closed form serves for the European value; only early exercise differs.
euro96 = bs_put(S0 * (1 - DELTA), K, T, r, SIG)
prem_lump = crr_scaled(2000, delta=DELTA) - crr_scaled(2000, delta=DELTA, american=False)
prem_yield = crr_scaled(2000, q=q_equiv) - crr_scaled(2000, q=q_equiv, american=False)

show("European, either form of the 4%", euro96, 4.4810, tol=1e-4)
show("American, 4% proportional at t_d", euro96 + prem_lump, 4.6223, tol=2e-4)
show("American, 4% as a continuous yield", euro96 + prem_yield, 4.4812, tol=2e-4)
show("early-exercise premium, lump at t_d", prem_lump, 0.1414, tol=5e-4)
show("early-exercise premium, continuous", prem_yield, 0.0002, tol=5e-4)
print()
print("form:   %+.1f%%  (proportional against the 2.00 of cash)"
      % (100 * ((euro96 + prem_lump) / correct - 1)))
print("timing: %+.1f%%  (the lump against the same 4%% spread out)"
      % (100 * ((euro96 + prem_lump) / (euro96 + prem_yield) - 1)))


## 4. Monte Carlo convergence — Example 20.3

The standard error falls by $\sqrt{10}$ per decade, exactly as predicted, and the
actual error bounces around inside it. A million paths buys half a cent on a
\$3.58 option; a 200-step tree was already inside a cent. For a vanilla
European option Monte Carlo is the wrong tool.

In [ ]:
def mc_put(n_paths, seed=1):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_paths)
    S_T = S0 * np.exp((r - 0.5 * SIG ** 2) * T + SIG * sqrt(T) * z)
    pay = np.maximum(K - S_T, 0.0) * exp(-r * T)
    return pay.mean(), pay.std() / sqrt(n_paths)


print("%12s %12s %16s %14s" % ("paths", "estimate", "standard error", "actual error"))
for n in (1_000, 10_000, 100_000, 1_000_000):
    m, se = mc_put(n)
    print("%12s %12.5f %16.5f %+14.5f" % ("{:,}".format(n), m, se, m - EXACT))
print()
print("a 200-step tree errs by %+.5f, in a fraction of the time" % (crr(200) - EXACT))

## 5. The control variate worth 58 million paths — Example 20.6

An arithmetic-average Asian has no formula; the *geometric*-average version does,
because the geometric average of lognormals is lognormal. The two payoffs are
almost perfectly correlated, so the geometric version prices the arithmetic one's
error away.

It is the pattern that gave the tree its accuracy in Example 20.1.

In [ ]:
S0A, KA, rA, sigA, TA, NFIX = 50.0, 50.0, 0.05, 0.30, 1.0, 12
t_fix = np.arange(1, NFIX + 1) * TA / NFIX


def geometric_asian_closed_form():
    mu = log(S0A) + (rA - 0.5 * sigA ** 2) * t_fix.mean()
    var = sigA ** 2 / NFIX ** 2 * sum((2 * i - 1) * t_fix[NFIX - i] for i in range(1, NFIX + 1))
    sg = sqrt(var)
    d1 = (mu - log(KA) + var) / sg
    return exp(-rA * TA) * (exp(mu + 0.5 * var) * N(d1) - KA * N(d1 - sg))


GEO = geometric_asian_closed_form()
show("geometric Asian, closed form", GEO, 4.012352)


def asian_mc(n_paths=100_000, seed=5):
    rng = np.random.default_rng(seed)
    dt = TA / NFIX
    z = rng.standard_normal((n_paths, NFIX))
    S = S0A * np.exp(np.cumsum((rA - 0.5 * sigA ** 2) * dt + sigA * sqrt(dt) * z, axis=1))
    arith = np.maximum(S.mean(axis=1) - KA, 0.0) * exp(-rA * TA)
    geo = np.maximum(np.exp(np.log(S).mean(axis=1)) - KA, 0.0) * exp(-rA * TA)
    beta = np.cov(arith, geo)[0, 1] / geo.var()
    cv = arith - beta * (geo - GEO)
    rho = np.corrcoef(arith, geo)[0, 1]
    return arith, cv, rho, beta


arith, cv, rho, beta = asian_mc()
n = len(arith)
print()
print("%-28s %12s %14s" % ("", "estimate", "standard error"))
print("%-28s %12.5f %14.5f" % ("plain Monte Carlo", arith.mean(), arith.std() / sqrt(n)))
print("%-28s %12.5f %14.5f" % ("geometric control variate", cv.mean(), cv.std() / sqrt(n)))
print()
print("correlation rho = %.4f, beta = %.4f" % (rho, beta))
factor = arith.var() / cv.var()
print("variance falls by a factor of %.0f" % factor)
print("plain Monte Carlo would need %.0f million paths for the same standard error"
      % (n * factor / 1e6))

## 6. A relative-TSR award — Example 20.7

Units earned depend on the company's return against an index; settlement is in
shares. The naive calculation $E[u]\,E[S_T]$ understates the value by 30% (the
correct value is 44% higher), because units are earned precisely on the paths where
the share price is high. The gap is entirely the covariance.

In [ ]:
def tsr_award(n_paths=400_000, seed=9):
    S0T, sig_c, sig_i, rho_ci, rT, TT = 50.0, 0.35, 0.20, 0.60, 0.05, 3.0
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n_paths)
    z2 = rho_ci * z1 + sqrt(1 - rho_ci ** 2) * rng.standard_normal(n_paths)
    R_c = np.exp((rT - 0.5 * sig_c ** 2) * TT + sig_c * sqrt(TT) * z1)
    R_i = np.exp((rT - 0.5 * sig_i ** 2) * TT + sig_i * sqrt(TT) * z2)
    rel = R_c - R_i                                   # relative total shareholder return
    units = np.clip(1.0 + rel / 0.30, 0.0, 2.0)       # 0% at -30%, 100% at 0, cap 200% at +30%
    S_T = S0T * R_c
    return units, S_T, exp(-rT * TT)


units, S_T, disc = tsr_award()
show("expected units earned", units.mean(), 0.8361, tol=2e-2)
show("expected share price", S_T.mean(), 58.1795, tol=2e-2)
show("naive product, disc * E[u] * E[S_T]", disc * units.mean() * S_T.mean(), 41.8704, tol=3e-2)
show("correct value, disc * E[u * S_T]", disc * (units * S_T).mean(), 60.2468, tol=3e-2)
print()
print("the correct value exceeds the naive one by %.0f%%"
      % (100 * ((units * S_T).mean() / (units.mean() * S_T.mean()) - 1)))
print("and the award is worth more than one share (50.00)")

## 7. One path, run backward — Example 20.8

Take the chapter's call ($S_0 = K = 100$, $r = 10\%$, $\sigma = 20\%$, $T = 1$) and a
single draw $\varepsilon = 0.5$. The forward pass stores every intermediate value; the
backward pass starts from $\bar V = 1$ and multiplies by one local derivative per step,
so each input reads its sensitivity off the step where it entered. The last lines bump
each input instead: four more pricings to get what one backward pass already gave.

In [ ]:
def call_aad(S0c, K_c, r_c, sig_c, T_c, eps):
    """One path of the call: a forward pass that stores, then one backward pass."""
    # forward pass: keep every intermediate value
    x = (r_c - 0.5 * sig_c ** 2) * T_c + sig_c * sqrt(T_c) * eps
    S_Tc = S0c * exp(x)
    disc_c = exp(-r_c * T_c)
    V = disc_c * max(S_Tc - K_c, 0.0)
    # backward pass: start from V_bar = 1, one local derivative per step
    V_bar = 1.0
    S_T_bar = V_bar * disc_c * (1.0 if S_Tc > K_c else 0.0)
    x_bar = S_T_bar * S_Tc
    grads = {
        "S0": S_T_bar * S_Tc / S0c,
        "sigma": x_bar * (sqrt(T_c) * eps - sig_c * T_c),
        "r": x_bar * T_c - T_c * V,           # r enters twice: in the drift and in the discount
        "K": -S_T_bar,
    }
    return x, S_Tc, V, S_T_bar, x_bar, grads


EPS1 = 0.5
BASE = dict(S0c=100.0, K_c=100.0, r_c=0.10, sig_c=0.20, T_c=1.0, eps=EPS1)
x1, ST1, V1, ST_bar, x_bar, g = call_aad(**BASE)
show("x", x1, 0.18, tol=5e-4)
show("S_T", ST1, 119.72, tol=5e-4)
show("V", V1, 17.84, tol=5e-4)
show("adjoint of S_T", ST_bar, 0.9048, tol=5e-4)
show("adjoint of x", x_bar, 108.33, tol=5e-4)
show("dV/dS0", g["S0"], 1.083, tol=5e-4)
show("dV/dsigma", g["sigma"], 32.50, tol=5e-4)
show("dV/dr", g["r"], 90.48, tol=5e-4)
show("dV/dK", g["K"], -0.905, tol=5e-4)

# The same four numbers by bump-and-revalue: one more pricing per input.
h = 1e-6
print()
print("%-8s %14s %14s" % ("input", "adjoint", "bump"))
for name, key in (("S0", "S0c"), ("sigma", "sig_c"), ("r", "r_c"), ("K", "K_c")):
    up = dict(BASE)
    up[key] += h
    bump = (call_aad(**up)[2] - V1) / h
    print("%-8s %14.6f %14.6f" % (name, g[name], bump))
print("one backward pass gave all four; bumping took four more pricings")

## 8. American Monte Carlo, step by step — Example 20.9

A simulated path standing at an intermediate date does not know its own future,
yet an early-exercise decision needs exactly that. Longstaff–Schwartz estimates
the continuation value by **regressing across paths**: many paths pass near the
same state, so the cross-section carries the conditional expectation.

Section 20.11's four steps, working backward from expiration:

1. take only the paths currently **in the money**, since elsewhere the decision is trivial;
2. regress the realized discounted continuation cash flow on $1, S, S^2$;
3. treat the fitted value as the continuation value and exercise where the immediate payoff beats it;
4. carry the cash flows back one step and repeat.

The next three cells do this once slowly, then look at what the regression
actually decided, then run it properly.

In [ ]:
def lsm_paths(n_paths, n_dates, seed=13):
    """Antithetic GBM paths on the exercise grid. Column i is date (i+1)*dt."""
    rng = np.random.default_rng(seed)
    dt = T / n_dates
    z = rng.standard_normal((n_paths, n_dates))
    z = np.concatenate([z, -z])
    return S0 * np.exp(np.cumsum((r - 0.5 * SIG ** 2) * dt + SIG * sqrt(dt) * z, axis=1)), dt


def longstaff_schwartz(S, dt, report_at=None):
    """The four steps of Section 20.11. Returns the value, and optionally the
    diagnostics from one exercise date so the regression can be inspected."""
    n_dates = S.shape[1]
    cash = np.maximum(K - S[:, -1], 0.0)              # value at expiry
    boundary, snapshot = np.full(n_dates, np.nan), None
    for i in range(n_dates - 2, -1, -1):
        cash *= exp(-r * dt)                          # step 4: carry back one step
        exercise = np.maximum(K - S[:, i], 0.0)
        itm = exercise > 0                            # step 1: in-the-money paths only
        if itm.sum() > 10:
            x = S[itm, i]
            design = np.vstack([np.ones_like(x), x, x * x]).T
            coef, *_ = np.linalg.lstsq(design, cash[itm], rcond=None)   # step 2
            continuation = design @ coef
            take = exercise[itm] > continuation       # step 3
            idx = np.where(itm)[0][take]
            cash[idx] = exercise[idx]
            if take.any():
                boundary[i] = x[take].max()           # highest S at which exercise wins
            if report_at is not None and i == report_at:
                snapshot = dict(i=i, S=x, cash=cash[itm].copy(), coef=coef,
                                continuation=continuation, exercise=exercise[itm],
                                take=take)
    cash *= exp(-r * dt)
    # Antithetic paths come in pairs (row i and row i + half), so the standard
    # error counts each pair once rather than treating the two as independent.
    half = len(cash) // 2
    pairs = 0.5 * (cash[:half] + cash[half:])
    return cash.mean(), pairs.std() / sqrt(half), boundary, snapshot


S_paths, dt = lsm_paths(20_000, 25)
value, se, boundary, snap = longstaff_schwartz(S_paths, dt, report_at=12)
print("value %.5f   s.e. %.5f   tree benchmark %.5f" % (value, se, AMER))

### What the regression decided, at one date

Date 13 of 25, roughly halfway. The fitted quadratic is the estimated continuation
value; the straight line $K - S$ is the immediate payoff. Where the payoff rises
above the fitted curve, the method exercises, and the crossing point is the
exercise boundary at that date.

Note how few paths carry the decision: only the in-the-money ones are regressed,
which is Longstaff and Schwartz's central device. Including the rest would spend
the fit's flexibility on a region where the answer is already known.

Watch the deepest row in the table. There the fit says *hold* when exercising is
plainly right: a quadratic has little data that far out and extrapolates badly.
It costs almost nothing here, because few paths reach it, but it is the reason
practitioners add basis functions or regress on the payoff itself.

In [ ]:
a, b, c = snap["coef"]
print("date %d of %d, t = %.3f" % (snap["i"] + 1, S_paths.shape[1], (snap["i"] + 1) * dt))
print("paths in the money: %s of %s" % ("{:,}".format(len(snap["S"])),
                                        "{:,}".format(len(S_paths))))
print("fitted continuation  E[V | S] = %.4f %+.4f S %+.6f S^2" % (a, b, c))
print("exercised on %s of them" % "{:,}".format(int(snap["take"].sum())))
print()
print("%8s %14s %16s %12s" % ("S", "exercise now", "fitted continue", "decision"))
order = np.argsort(snap["S"])
for j in order[::max(1, len(order) // 12)]:
    print("%8.2f %14.4f %16.4f %12s"
          % (snap["S"][j], snap["exercise"][j], snap["continuation"][j],
             "EXERCISE" if snap["take"][j] else "hold"))

# the two curves cross where c S^2 + (b+1) S + (a-K) = 0. A quadratic fit crosses
# a line twice; the boundary is the LARGER root -- the highest S at which
# exercising still beats holding.
roots = np.roots([c, b + 1.0, a - K])
roots = sorted(x.real for x in roots if abs(x.imag) < 1e-9 and 0 < x.real <= K)
print()
print("crossings of the two curves: %s" % ", ".join("%.2f" % x for x in roots))
print("the boundary is the larger, S = %.2f: exercise below it, hold above it" % roots[-1])
print("the paths agree -- highest S actually exercised was %.2f" % boundary[snap["i"]])

### The boundary it traces, and the bias it carries

Two things are worth seeing. The regression recovers an **exercise boundary** rising
toward the strike as expiration approaches: the same curve a tree locates node by
node, found here from paths alone.

And there are *two* biases pulling opposite ways. Fitting the rule and valuing on
the **same** paths lets the rule use information those paths should not have,
which pushes the answer up; an imperfect exercise rule pushes it down. Desks
separate the two by valuing on fresh paths. Measure the gap rather than assume
it: with three basis functions and 40,000 paths the foresight bias turns out to
be negligible here: the two values below differ by a small fraction of a
standard error.

In [ ]:
print("exercise boundary traced by the regression")
print("%10s %12s" % ("t", "boundary S"))
for i in range(0, S_paths.shape[1] - 1, 4):
    if not np.isnan(boundary[i]):
        print("%10.3f %12.2f" % ((i + 1) * dt, boundary[i]))

print()
print("in-sample against out-of-sample")
S_fit, _ = lsm_paths(20_000, 25, seed=13)
S_new, _ = lsm_paths(20_000, 25, seed=99)
in_sample, se_in, _, _ = longstaff_schwartz(S_fit, dt)
out_sample, se_out, _, _ = longstaff_schwartz(S_new, dt)
print("  fitted and valued on the same paths  %.5f  (%+.5f)" % (in_sample, in_sample - AMER))
print("  valued on independent paths          %.5f  (%+.5f)" % (out_sample, out_sample - AMER))
print("  tree benchmark                       %.5f" % AMER)

### Example 20.10's table

With 25 or 50 exercise dates the method prices a Bermudan option, so each
estimate is measured against a tree that may exercise only on those dates. Any
exercise rule other than the optimal one earns less, so the method is biased
downward, but at these path counts the bias is of the order of a standard error:
a single row cannot prove it, and the rows scatter by a standard error or more
either way. More paths shrink the standard error but not the bias; a better basis
reduces the bias, and more exercise dates move the target toward the American.
The path counts are counted as the book counts them, in antithetic pairs, and the
standard error counts each pair once.

For a one-dimensional American option this is strictly the wrong choice: the
200-step tree beats it in a fraction of the time. Its value is that it works
where a tree cannot be built.

In [ ]:
def bermudan(n_dates, per_date):
    """Put exercisable only on the n_dates exercise dates, on a CRR tree."""
    n = n_dates * per_date
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    v = np.maximum(K - S0 * u ** (2 * np.arange(n + 1) - n), 0.0)
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if i > 0 and i % per_date == 0:
            v = np.maximum(v, K - S0 * u ** (2 * np.arange(i + 1) - i))
    return float(v[0])


# Richardson between 80 and 160 tree steps per exercise date
BERM = {nd: 2 * bermudan(nd, 160) - bermudan(nd, 80) for nd in (25, 50)}
show("Bermudan put, 25 exercise dates", BERM[25], 3.6907)
show("Bermudan put, 50 exercise dates", BERM[50], 3.6938)
print()
print("%10s %8s %12s %12s %12s %12s"
      % ("paths", "dates", "estimate", "std error", "Bermudan", "error"))
for n_paths, n_dates in ((20_000, 25), (100_000, 25), (100_000, 50), (400_000, 50)):
    S_i, dt_i = lsm_paths(n_paths // 2, n_dates)      # n_paths in antithetic pairs
    m, s, _, _ = longstaff_schwartz(S_i, dt_i)
    print("%10s %8d %12.5f %12.4f %12.4f %+12.4f"
          % ("{:,}".format(n_paths), n_dates, m, s, BERM[n_dates], m - BERM[n_dates]))
print()
print("a 200-step tree gives %+.5f in a fraction of the time"
      % (crr(200, american=True) - AMER))

## 9. What instability looks like — Example 20.11

Refining the space grid while leaving the time grid alone destroys the
calculation. The condition is roughly $\Delta t \lesssim 1/(\sigma^2 M^2)$, so
**doubling the space resolution needs four times as many time steps**.

There is one consolation: an unstable explicit scheme fails *loudly*. The exact
magnitude of the blow-up depends on the boundary handling; what is reproducible is
that it is astronomically wrong rather than slightly wrong.

In [ ]:
def fd_put(M, n_t, theta, Smax=200.0, american=True):
    """theta = 0 explicit, 1 implicit, 0.5 Crank-Nicolson."""
    dS, dt = Smax / M, T / n_t
    S = np.linspace(0.0, Smax, M + 1)
    v = np.maximum(K - S, 0.0)
    j = np.arange(1, M)
    a = 0.5 * dt * (SIG ** 2 * j ** 2 - r * j)
    b = -dt * (SIG ** 2 * j ** 2 + r)
    c = 0.5 * dt * (SIG ** 2 * j ** 2 + r * j)
    if theta == 0.0:
        for _ in range(n_t):
            vn = v.copy()
            v[1:M] = a * vn[0:M - 1] + (1 + b) * vn[1:M] + c * vn[2:M + 1]
            v[0], v[M] = K, 0.0
            if american:
                v = np.maximum(v, K - S)
        return float(np.interp(S0, S, v))
    A = np.diag(1 - theta * b)
    B = np.diag(1 + (1 - theta) * b)
    for i in range(M - 1):
        if i > 0:
            A[i, i - 1], B[i, i - 1] = -theta * a[i], (1 - theta) * a[i]
        if i < M - 2:
            A[i, i + 1], B[i, i + 1] = -theta * c[i], (1 - theta) * c[i]
    for _ in range(n_t):
        rhs = B @ v[1:M]
        rhs[0] += a[0] * K
        v[1:M] = np.linalg.solve(A, rhs)
        v[0], v[M] = K, 0.0
        if american:
            v = np.maximum(v, K - S)
    return float(np.interp(S0, S, v))


print("%12s %12s %20s" % ("space steps", "time steps", "value"))
for M, n_t in ((100, 1000), (200, 1000), (200, 4000)):
    val = fd_put(M, n_t, 0.0)
    print("%12d %12d %20s" % (M, n_t, ("%.4f" % val) if abs(val) < 1e6 else "%.0e" % val))
print()
print("the middle row is not slightly wrong; it is astronomically wrong")

## 10. Implicit against Crank–Nicolson — Example 20.12

Implicit errors fall by about two per refinement, which is first order.
Crank–Nicolson's fall by about three and a half, then three: its symmetric
time-averaging is second order on the smooth part of the problem, and the
exercise condition imposed after each solve is first order, which shows as the
grid is refined. Both converge from below, because a coarse grid cannot
represent the exercise boundary exactly.

In [ ]:
print("%8s %8s %14s %18s %14s" % ("space", "time", "implicit", "Crank-Nicolson", "CN error"))
for M, n_t, wi, wc in ((100, 50, 3.65776, 3.67363), (200, 100, 3.68196, 3.69033),
                       (400, 200, 3.69056, 3.69494)):
    imp, cn = fd_put(M, n_t, 1.0), fd_put(M, n_t, 0.5)
    print("%8d %8d %14.5f %18.5f %+14.5f    book %.5f / %.5f"
          % (M, n_t, imp, cn, cn - AMER, wi, wc))

## 11. Greeks from the tree and from the grid — Example 20.13

Delta, gamma and theta cost nothing extra: each method has already
computed the neighboring values they are differences of. Two
implementations sharing no arithmetic should agree, which is the
cheapest check available on either of them. Vega and rho are the
exception, and the cell shows what they cost.


In [ ]:
# Delta, gamma and theta cost nothing extra: both methods have already
# computed the neighboring values they are differences of. On a tree, the
# nodes at steps one and two; on a grid, the neighbors of S0 in the t = 0
# column and the column beside it.

def tree_greeks(n):
    """Value, delta, gamma and theta from the first two steps of a CRR tree."""
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    v = np.maximum(K - S0 * u ** (2 * np.arange(n + 1) - n), 0.0)
    keep = {}
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:i + 2] + (1.0 - p) * v[:i + 1])
        v = np.maximum(v, K - S0 * u ** (2 * np.arange(i + 1) - i))
        if i <= 2:
            keep[i] = v.copy()
    suu, sdd = S0 * u * u, S0 * d * d
    delta = (keep[1][1] - keep[1][0]) / (S0 * u - S0 * d)
    gamma = ((keep[2][2] - keep[2][1]) / (suu - S0)
             - (keep[2][1] - keep[2][0]) / (S0 - sdd)) / (0.5 * (suu - sdd))
    theta = (keep[2][1] - keep[0][0]) / (2.0 * dt)
    return float(keep[0][0]), float(delta), float(gamma), float(theta)


def fd_greeks(M, n_t, theta=0.5, Smax=200.0, sig=SIG, rate=r):
    """fd_put, keeping the last two columns so the Greeks can be read off."""
    dS, dt = Smax / M, T / n_t
    S = np.linspace(0.0, Smax, M + 1)
    payoff = np.maximum(K - S, 0.0)
    v = payoff.copy()
    j = np.arange(1, M)
    a = 0.5 * dt * (sig ** 2 * j ** 2 - rate * j)
    b = -dt * (sig ** 2 * j ** 2 + rate)
    c = 0.5 * dt * (sig ** 2 * j ** 2 + rate * j)
    A = np.diag(1 - theta * b)
    B = np.diag(1 + (1 - theta) * b)
    for i in range(M - 1):
        if i > 0:
            A[i, i - 1], B[i, i - 1] = -theta * a[i], (1 - theta) * a[i]
        if i < M - 2:
            A[i, i + 1], B[i, i + 1] = -theta * c[i], (1 - theta) * c[i]
    prev = v.copy()
    for _ in range(n_t):
        rhs = B @ v[1:M]
        rhs[0] += a[0] * K
        prev = v.copy()
        v = v.copy()
        v[1:M] = np.linalg.solve(A, rhs)
        v[0], v[M] = K, 0.0
        v = np.maximum(v, payoff)
    j0 = int(round(S0 / dS))
    delta = (v[j0 + 1] - v[j0 - 1]) / (2.0 * dS)
    gamma = (v[j0 + 1] - 2.0 * v[j0] + v[j0 - 1]) / dS ** 2
    return (float(v[j0]), float(delta), float(gamma),
            float((prev[j0] - v[j0]) / dt))


print("%-26s %10s %10s %10s %10s"
      % ("method", "value", "delta", "gamma", "theta"))
for label, got in (("tree, 6,000 steps", tree_greeks(6000)),
                   ("Crank-Nicolson, 800x400", fd_greeks(800, 400))):
    print("%-26s %10.4f %10.4f %10.5f %10.4f" % ((label,) + got))

d1 = (log(S0 / K) + (r + 0.5 * SIG ** 2) * T) / (SIG * sqrt(T))
euro_gamma = exp(-0.5 * d1 * d1) / sqrt(2 * np.pi) / (S0 * SIG * sqrt(T))
print("%-26s %10s %10.4f %10.5f"
      % ("European, closed form", "", N(d1) - 1.0, euro_gamma))

# Volatility and the rate are parameters, not state variables, so neither is
# a difference of grid values: each costs a second solve, here on the
# 800 x 400 grid the example tabulates.
base = fd_greeks(800, 400)[0]
print()
print("vega, per volatility point   %+.4f"
      % (fd_greeks(800, 400, sig=SIG + 0.01)[0] - base))
print("rho,  per basis point        %+.4f"
      % (fd_greeks(800, 400, rate=r + 0.0001)[0] - base))
